In [ ]:
%matplotlib notebook
from rfsoc_rfdc.rfsoc_overlay import RFSoCOverlay
from rfsoc_rfdc.overlay_task import OverlayTask
from rfsoc_rfdc.overlay_task import BlinkLedTask
from rfsoc_rfdc.beamformer_task import BeamformerTxTask, BeamformerRxTask

from rfsoc_rfdc.transmitter.multi_ch_tx_mimo_task import MultiChTxMIMOTask
from rfsoc_rfdc.receiver.multi_ch_rx_mimo_task import MultiChRxMIMOTask

from rfsoc_rfdc.rfdc_task import RfdcTask 
from rfsoc_rfdc.mts_task import MtsTask
from rfsoc_rfdc.array_calib_task import ArrayCalibTask
from rfsoc_rfdc.overlay_task import OverlayTask, TASK_STATE

from rfsoc_rfdc.rfdc_config import ZCU216_CONFIG

import sys
import os
import time

In [ ]:
from rfsoc_rfdc.dsp.ofdm import OFDM
from rfsoc_rfdc.dsp.mimo_detection import MIMODetection

In [ ]:
ol = RFSoCOverlay(path_to_bitstream="./rfsoc_rfdc/bitstream/rfsoc_rfdc_v47_8t2r_bf.bit")
NEW_CONFIG = {
    "RefClockForPLL": 300.0,
    "DACSampleRate": 2400.0,
    "DACInterpolationRate": 4,
    "DACNCO": 700,
    "ADCSampleRate": 2400.0,
    "ADCInterpolationRate": 4,
    "ADCNCO": -700
}
ZCU216_CONFIG.update(NEW_CONFIG)

In [ ]:
rfdc_t = RfdcTask(ol, debug_mode=True, board="ZCU216")
mts_t = MtsTask(ol, board="ZCU216", debug_mode=True)

for task in [mts_t, rfdc_t]:
    task.start()
    task.join()

In [ ]:
calib_t = ArrayCalibTask(ol, num_tx_ch=2, num_dacs=8, num_rx_ch=2, num_adcs=2)
calib_t.start()
calib_t.join()

In [ ]:
true_samp_rate = ZCU216_CONFIG['DACSampleRate'] / ZCU216_CONFIG['DACInterpolationRate'] * 1e6

tx_bf_task = BeamformerTxTask(ol, num_channels=8, debug_mode=True)
tx_bf_task.calib_steer(0)

In [ ]:
led_t = BlinkLedTask(ol)
led_t.start()

for atten in range(0, 31, 6):

    for mcs in [0, 2, 4, 7]:
        
        ZCU216_CONFIG['OFDM_ATTEN_DB'] = atten
        ZCU216_CONFIG['DETECTION_SCHEME'] = MIMODetection(sample_rate=true_samp_rate, tx_num=2, rx_num=2, MCS=mcs)
        ZCU216_CONFIG['CONFIG_NAME'] = "ATTEN_" + str(atten) + "_CHARM_OTA_2T2R_MCS_" + str(mcs)

        print(f"MCS={mcs}, OFDM_ATTEN_DB={atten}")

        tx_t = MultiChTxMIMOTask(ol, mode="iq2real", channel_count=2, dp_vect_dim=4)
        tx_t.start()

        time.sleep(3)
        
        rx_t = MultiChRxMIMOTask(ol, mode="real2iq", channel_count=2, dp_vect_dim=4)
        rx_t.start()
        
        while rx_t.task_state != TASK_STATE["STOP"]:
            time.sleep(1)
        
        tx_t.stop()
        rx_t.stop()

led_t.stop()